# Cross-branch DPO-delta transfer — coefficient sweep, reciprocal direction, judges

Three jobs, in order. **Every section is independently runnable and resumable** — the
runner skips a unit whose output file already exists, and shards checkpoint inside a
unit. If a Colab session dies, re-upload the last results zip (section 5) and continue.

| Section | What | Units | Rough T4 time |
|---|---|---|---|
| A | A→B coefficient sweep at 0.5 and 2.0 | 12 | ~90 min |
| B | B→A reciprocal: Stage-1 gate + Stage-2 core | 8 + 6 | ~105 min |
| C | StrongREJECT + WildGuard judges on quadrants A/C | — | ~45–90 min |

Total is more than one free session. **Plan on two or three**, packaging and
downloading after each section.

**Why the reciprocal (B→A) matters:** running only Alpaca→Dolly leaves open that
whatever moved the target is a property of *Dolly's* pre-DPO model rather than of the
transferred delta. Swapping the roles is the control for that.

**Why the judges:** the rule-based four-way classifier stays the primary endpoint, but
StrongREJECT/WildGuard give a continuous harmfulness reading that does not depend on
refusal-phrase regexes. Section C scores quadrants A and C only — they are harmfulness
judges and say nothing useful about the benign quadrants, where over-refusal is already
a rule-based endpoint.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = 'c784139b156910b717a73ae7ae5e1f2cca803c98'

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
print("checked out", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2b. HF authentication — REQUIRED for section C

The judge models (`qylu4156/strongreject-15k-v1`, `allenai/wildguard`) are gated repos:
without a token section C fails with 401. Sections A and B only need it to avoid
unauthenticated Hub rate limits across many model loads.

Set the `HF_TOKEN` Colab secret first (key icon, left sidebar), then run this cell.
Setting `os.environ` is what matters — it is inherited by every `!python -m ...`
subprocess, whereas `login()` alone writes into HF_HOME and can be orphaned.

In [ ]:
import os

try:
    from google.colab import userdata
    from huggingface_hub import login
    _tok = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = _tok
    login(token=_tok)
    print('HF login OK')
except Exception as e:
    print('HF NOT authenticated:', repr(e))
    print('Sections A/B will still run (slower). Section C WILL FAIL: the judge '
          'repos are gated. Set the HF_TOKEN Colab secret and rerun this cell.')

## 3. Apply the crossbranch patch

Upload `crossbranch_p0_patch.zip` from Downloads. This build adds the reciprocal-direction
guards and the judge modules.

In [ ]:
from google.colab import files
import zipfile, io

uploaded = files.upload()
assert len(uploaded) == 1, "upload exactly one file: crossbranch_p0_patch.zip"
name, data = next(iter(uploaded.items()))
with zipfile.ZipFile(io.BytesIO(data)) as z:
    names = z.namelist()
    assert all(n.startswith(('src/analysis/crossbranch/', 'tests/analysis/crossbranch/'))
               for n in names), "patch contains files outside the crossbranch package"
    z.extractall('.')
print(f"applied {len(names)} files from {name}")

## 4. Dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install bitsandbytes
!pip uninstall -y torchao || true
!nvidia-smi

## 5. Restore previous results (skip on a first run)

Upload the newest `crossbranch_stage2_results*.zip`. This puts the completed Stage-1 and
A→B coef-1.0 outputs back in place so the runner **skips** them instead of regenerating,
and so section C can judge everything in one pass. Cancel the dialog to skip.

In [ ]:
import zipfile, io, os, glob
from google.colab import files

os.makedirs('results/crossbranch', exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extractall('results/crossbranch')
    print(f"restored {name}")

raw = [p for p in sorted(glob.glob('results/crossbranch/raw/crossbranch_*.json'))
       if not p.endswith('_binding.json')]
print(f"\n{len(raw)} raw response files present:")
for p in raw:
    print("  ", os.path.basename(p))

## 6. Copy activations and both direction vectors from Drive

In [ ]:
RESULTS_SOURCE_DIR = '/content/drive/MyDrive/dpo_v2/results'

import shutil
from pathlib import Path

src_root = Path(RESULTS_SOURCE_DIR)
assert (src_root / 'activations').exists(), f"{src_root}/activations not found"

STAGES = ('M2', 'M3', 'M2_alt', 'M3_alt')
ACT_SUFFIXES = ('_final.npy', '_pooled.npy', '_metadata.json', '_metadata_binding.json')
DIR_SUFFIXES = ('_v2_direction.npy', '_v2_direction_binding.json')

copied, missing = [], []
dst = Path('results/activations'); dst.mkdir(parents=True, exist_ok=True)
for stage in STAGES:
    for suf in ACT_SUFFIXES:
        s = src_root / 'activations' / f'{stage}{suf}'
        (copied if s.exists() else missing).append(s.name)
        if s.exists():
            shutil.copy2(s, dst / f'{stage}{suf}')

dst_dir = Path('results/refusal_direction'); dst_dir.mkdir(parents=True, exist_ok=True)
for stage in ('M3', 'M3_alt'):
    for suf in DIR_SUFFIXES:
        s = src_root / 'refusal_direction' / f'{stage}{suf}'
        (copied if s.exists() else missing).append(s.name)
        if s.exists():
            shutil.copy2(s, dst_dir / f'{stage}{suf}')

print(f"copied {len(copied)} files")
if missing:
    print(f"MISSING ({len(missing)}):")
    for m in missing:
        print("  ", m)

## 7. Preflight and test gate

In [ ]:
import json
from pathlib import Path
import numpy as np
from src.v2_io import load_run_inputs, identity_snapshot, load_json

bp, bsha, sp, ssha = load_run_inputs(None, None, 'logs/direction_split_manifest.json')
rows = [json.loads(l) for l in Path(bp).read_text(encoding='utf-8').splitlines() if l.strip()]
snap = identity_snapshot(rows)
print(f"benchmark {bsha[:12]}...  split {ssha[:12]}...  rows {len(rows)}\n")

ok = True
for stage in ('M2', 'M3', 'M2_alt', 'M3_alt'):
    arr = np.load(f'results/activations/{stage}_final.npy', mmap_mode='r')
    good = (arr.shape[0] == len(rows)
            and load_json(f'results/activations/{stage}_metadata.json') == snap
            and load_json(f'results/activations/{stage}_metadata_binding.json').get('benchmark_sha256') == bsha)
    ok &= good
    print(f"  activation {stage:8s} {str(arr.shape):18s} {'PASS' if good else 'FAIL'}")

for stage in ('M3', 'M3_alt'):
    d = np.load(f'results/refusal_direction/{stage}_v2_direction.npy')
    norm = float(np.linalg.norm(d[24]))
    good = abs(norm - 1.0) < 1e-3
    ok &= good
    print(f"  direction  {stage:8s} layer24 norm={norm:.6f}  {'PASS' if good else 'FAIL'}")

assert ok, "preflight FAILED - do not proceed"
print("\nAll PASS.")

In [ ]:
!python -m pytest tests/analysis/crossbranch -q

---
# Section A — A→B coefficient sweep (0.5 and 2.0)

Stage 2 ran at coefficient 1.0 only, per the plan's staging: 1.0 first, then 0.5 and 2.0
**only if the 1.0 run is stable**. It was stable — degeneracy stayed under 3.5% in every
quadrant, no collapse — so the sweep is the sanctioned next step.

What it buys: a dose–response curve per arm. If identity and the within-quadrant shuffle
stay indistinguishable at every coefficient, that is far stronger evidence that the
movement is carried by a shared per-quadrant component than a single dose can give. If
they separate at some dose, that is the prompt-conditioned effect showing up.

In [ ]:
!python -m src.analysis.crossbranch.delta --stage2

In [ ]:
STAGE2_CORE = (
    "xfer_delta_source_identity xfer_delta_source_shuf_wq "
    "xfer_delta_source_normmatched xfer_delta_source_dosematched "
    "dir_source_matched dir_target_matched"
)
!python -m src.analysis.crossbranch.runner --dry-run --allow-stage2 \
    --conditions {STAGE2_CORE} --coefficients 0.5 2.0

In [ ]:
!python -m src.analysis.crossbranch.runner --allow-stage2 \
    --conditions {STAGE2_CORE} --coefficients 0.5 2.0

---
# Section B — the reciprocal direction (B→A)

Roles swap: source is now **Dolly** (Δ_B = M3_alt − M2_alt), target is **Alpaca** (inject
into M2, compare against M3).

**Two collision hazards, both now guarded in code — do not work around them:**

1. Delta artifact filenames are direction-neutral (`delta_source_L24.npz` means "the
   source branch's delta"). Assembling B→A into the A→B directory would overwrite
   `delta_source` with Δ_B while every condition name stayed identical — the run would
   succeed and the numbers would be wrong. So B→A gets its own `--deltas-dir`, and
   `delta.py` refuses to cross-assemble without `--force-direction`.
2. Shard unit keys now carry the direction tag. Without that a B→A unit would find A→B's
   completed shards for the same (condition, coefficient), skip generation entirely, and
   merge the wrong branch's rows out under the reciprocal filename.

Stage 1 runs first, at all three coefficients — the same hard gate as before, now asking
whether **Alpaca's own** delta reproduces **Alpaca's own** post-DPO behaviour. A B→A
Stage-2 result is not interpretable without it: a null would otherwise conflate "the delta
does not transfer" with "this site is not causally sufficient in this branch".

In [ ]:
DELTAS_BA = 'results/crossbranch/deltas_BtoA'
!python -m src.analysis.crossbranch.delta --stage2 \
    --source-branch B --target-branch A --out-dir {DELTAS_BA}

In [ ]:
!python -m src.analysis.crossbranch.runner --dry-run \
    --source-branch B --target-branch A --deltas-dir {DELTAS_BA}

In [ ]:
!python -m src.analysis.crossbranch.runner \
    --source-branch B --target-branch A --deltas-dir {DELTAS_BA}

### B.2 — reciprocal Stage-1 gate

Read this before running B.3. If the gate does not pass, the reciprocal Stage 2 is not
interpretable: stop and report the asymmetry rather than running it.

In [ ]:
!python -m src.analysis.crossbranch.analyze --source-branch B --target-branch A

### B.3 — reciprocal Stage 2 (RUN ONLY IF B.2's gate passed)

From cell B.2's output, read `mechanical_gate_passed`.

- **`True`**  -> run the guard cell then B.3.
- **`False`** -> **do not run B.3.** The reciprocal Stage 1 gate failing means a
  B->A Stage-2 null would be uninterpretable (can't tell "delta doesn't transfer"
  from "this site isn't causally sufficient in the Alpaca branch"). Package what
  you have (section "Package and download"), bring it back, and report the
  asymmetry. Stop here.

The next cell is a hard guard: it reads the B->A gate JSON and raises unless the
mechanical gate passed, so B.3 cannot run by accident.

In [ ]:
# Hard guard: B.3 below refuses to run unless the reciprocal Stage-1 gate passed.
import json, pathlib
_gate = pathlib.Path("results/crossbranch/analysis/crossbranch_BtoA_analysis.json")
assert _gate.exists(), "run cell B.2 (reciprocal gate) first"
_g = json.loads(_gate.read_text(encoding="utf-8"))["gate"]
print("reciprocal gate quadrant :", _g["gate_quadrant"])
print("mechanical_gate_passed   :", _g["mechanical_gate_passed"])
print("inconclusive_by_collapse :", _g["inconclusive_by_collapse"])
assert _g["mechanical_gate_passed"], (
    "reciprocal Stage-1 gate did NOT pass -- do not run reciprocal Stage 2. "
    "Package results, bring them back, report the A->B / B->A asymmetry."
)
print("
OK -- reciprocal gate passed; B.3 may run.")

In [ ]:
!python -m src.analysis.crossbranch.runner --allow-stage2 \
    --source-branch B --target-branch A --deltas-dir {DELTAS_BA} \
    --conditions {STAGE2_CORE} --coefficients 1.0

---
# Section C — judges (StrongREJECT + WildGuard)

Quadrants **A and C only**. The manifest builder writes quadrant-filtered copies of every
raw file into `results/crossbranch/judge_inputs/` and rewrites each row's `stage` to a
fully-qualified `direction|condition|coefN` key — necessary because the judge pipeline's
record schema drops `coef`, `source_branch` and `target_branch`, which would otherwise
make every arm indistinguishable in the output.

Run this **last**, after every generation section you intend to do, so one judge pass
covers everything. Judges load one at a time (4-bit), so peak VRAM is a single 7B model.

In [ ]:
!python -m src.analysis.crossbranch.build_judge_manifest

In [ ]:
!python -m src.analysis.behavioral_judges \
    --response-manifest results/crossbranch/manifests/crossbranch_judge_manifest.json \
    --out-dir results/crossbranch/judges --run-live --scope all

---
## Package and download

Safe to run after **any** section — do it after each one so a dead session never costs
more than the section in flight.

In [ ]:
import shutil, os
shutil.make_archive('/content/crossbranch_sweep_results', 'zip', 'results/crossbranch')
print(f"{os.path.getsize('/content/crossbranch_sweep_results.zip') / 1e6:.1f} MB")

from google.colab import files
files.download('/content/crossbranch_sweep_results.zip')

---
## STOP — analysis happens on the local machine

Bring `crossbranch_sweep_results.zip` back and unzip into `results/crossbranch/`.

### Which outputs must exist by completion

Depending on which sections you ran:

- **Section A (sweep)** — `results/crossbranch/raw/crossbranch_AtoB_<arm>_coef0.5.json`
  and `_coef2.json` for all 6 Stage-2 arms (+ `_binding.json` each), 414 rows each.
- **Section B (reciprocal)** —
  `results/crossbranch/deltas_BtoA/*.npz` + `crossbranch_deltas_binding.json`
  (roles source=B target=A);
  `results/crossbranch/raw/crossbranch_BtoA_baseline_target_coefna.json`,
  `_reference_target_coefna.json`, `_own_delta_target_coef{0.5,1,2}.json`,
  `_own_normmatched_random_coef{0.5,1,2}.json` (8 Stage-1 units);
  `results/crossbranch/analysis/crossbranch_BtoA_analysis.json` (the gate);
  and **only if the gate passed**, `crossbranch_BtoA_<arm>_coef1.json` for the 6 arms.
- **Section C (judges)** — `results/crossbranch/judges/behavioral_judges_v2_<ts>.json`
  with `judge_status.strong_reject == "scored"` and `.wildguard == "scored"`
  (or an explicit `unavailable: ...` string — not `not_run`).

### Then, locally (no GPU)

```
python -m src.analysis.crossbranch.analyze_stage2 --coef 0.5
python -m src.analysis.crossbranch.analyze_stage2 --coef 1.0
python -m src.analysis.crossbranch.analyze_stage2 --coef 2.0
python -m src.analysis.crossbranch.analyze_stage2 --source-branch B --target-branch A   # only if B.3 ran
python -m src.analysis.crossbranch.compare_directions                                    # only if B.3 ran
python -m src.analysis.crossbranch.analyze_judges --judge-file results/crossbranch/judges/behavioral_judges_v2_<ts>.json
python -m src.analysis.crossbranch.plot_stage2
```

What each answers:

- **`analyze_stage2` across coefficients** — does identity-vs-shuffle stay
  indistinguishable at every dose, or does a prompt-conditioned effect emerge at one?
- **`compare_directions`** — does the pattern hold with the branches swapped?
  Agreement -> property of the delta and the site; disagreement -> property of one
  branch's own pre-DPO model. Reports `underpowered` separately from `no`.
- **`analyze_judges`** — does a continuous harmfulness judge, independent of
  refusal-phrase regexes, agree with the rule-based reading? Check `coverage` per arm;
  a low-coverage endpoint is not confirmation.